Exploration Charts — GoEmotions (28 fixed labels) fork
==========================================================
Charts the GoEmotions (28 fixed labels) scores produced by `05.2_song_analysis_goemotions.ipynb`.

The companion notebook `06.1_exploration_charts_zeroshot.ipynb` runs the identical chart set over the other
classifier. Both forks are produced by `04_classification.ipynb` under one **shared
scoring contract** (see its header), so they differ only where they must:

| | zero-shot NLI (04 §4A → 05.1 → 06.1) | GoEmotions (04 §4B → 05.2 → 06.2) |
|---|---|---|
| scoring | independent per-label, [0, 1] | *same* |
| `unclassified` | no scoreable lyrics | *same* — drops the same ~107 songs |
| confidence | flagged at 0.30, never dropped | *same* |
| **labels** | **10, hand-picked for songs** | **28, fixed (Reddit-trained)** |
| **includes `neutral`** | **no** | **yes** |
| **model** | **`bart-large-mnli`, zero-shot** | **`roberta-base-go_emotions`, supervised** |
| **typical top score** | **~0.97** | **~0.50** |

The bold rows are the real, irreducible differences; everything above them used
to differ too, purely by config. The last row still matters for reading charts:
RoBERTa's sigmoids are calibrated systematically lower than bart-mnli's
entailment probabilities, so **a raw 0.4 does not mean the same thing in each
fork** even under the shared contract.

Absolute-magnitude charts below are therefore labelled fork-local. The **z-score
charts (§ 4) are the ones that survive a cross-fork read** — standardising
within a fork cancels the calibration offset and leaves only "which regions are
unusually high on this emotion, relative to this fork's own baseline".

`neutral` is charted out of the emotion grids (it has no counterpart in the
zero-shot taxonomy and would otherwise dominate every panel) and is reported on
its own below. The dominant-emotion charts use `dominant_emotion_emotive` —
the strongest non-neutral label — computed in `05.2`.

In [0]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
PROCESSED = PROJECT_ROOT / "data" / "processed"

# ── Fork config ───────────────────────────────────────────────────────────────
LABEL        = "GoEmotions (28 fixed labels)"
INPUT_PATH   = PROCESSED / "05.2_titles_emotion_scores_goemotions.csv"
DOMINANT_COL = "dominant_emotion_emotive"   # column charted as "the song's emotion"
EXCLUDE      = ["neutral"]          # labels held out of the emotion charts
TOP_N        = 12            # emotions charted; 28 labels is unreadable in a grid; the tail is near-zero

df = pd.read_csv(INPUT_PATH)

emotion_cols = [
    c for c in df.columns
    if c.startswith("emotion_") and c.replace("emotion_", "") not in EXCLUDE
]
# Charted subset: the strongest TOP_N emotions overall. With 28 labels a full
# grid is unreadable, so the tail is trimmed here rather than in every chart.
charted_cols = (
    df[emotion_cols].mean().sort_values(ascending=False).head(TOP_N).index.tolist()
)
charted_names = [c.replace("emotion_", "") for c in charted_cols]

print(f"{LABEL}: {len(df)} songs, {len(emotion_cols)} emotion columns "
      f"(excluding {EXCLUDE or 'none'}), charting top {len(charted_cols)}")
print(charted_names)
display(df.head())

In [0]:
# ── Palette ───────────────────────────────────────────────────────────────────
# Sequential = ONE hue light→dark (magnitude). Diverging = two poles + a neutral
# gray midpoint (polarity around zero). Never a rainbow ramp for either job — a
# rainbow makes non-adjacent values look adjacent and hides the actual ordering.
SEQ_BLUE = [
    "#cde2fb", "#b7d3f6", "#9ec5f4", "#86b6ef", "#6da7ec",
    "#5598e7", "#3987e5", "#2a78d6", "#256abf", "#1c5cab",
    "#184f95", "#104281", "#0d366b",
]
DIVERGING = [[0.0, "#0d366b"], [0.25, "#3987e5"], [0.5, "#f0efec"],
             [0.75, "#e34948"], [1.0, "#7d1f1e"]]

# Fixed categorical order — hues are assigned by slot and never cycled or
# re-ordered by rank, so a given region keeps its colour across every chart.
CATEGORICAL = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100",
               "#e87ba4", "#008300", "#4a3aa7", "#e34948"]

regions = sorted(df["region"].dropna().unique())
if len(regions) > len(CATEGORICAL):
    raise ValueError(
        f"{len(regions)} regions but only {len(CATEGORICAL)} validated hues. "
        "Fold the smallest into 'Other' or facet instead of generating a 9th hue."
    )
REGION_COLOR = dict(zip(regions, CATEGORICAL))

LAYOUT = dict(
    template="plotly_white",
    font=dict(family="Inter, -apple-system, Helvetica, sans-serif", size=13,
              color="#0b0b0b"),
    title_font_size=17,
    margin=dict(l=90, r=40, t=90, b=70),
)
print(f"{len(regions)} regions: {regions}")

### 1. Coverage and confidence

Only songs with no scoreable lyrics were dropped upstream in the 05.x join, and
both forks drop the same ~107 of them — so **this fork and its sibling are
charting the identical song set**, and per-region counts are directly
comparable between the two notebooks.

Low-confidence songs are *kept*, flagged rather than dropped. The flag uses the
same 0.30 bar in both forks, but because the two models are calibrated
differently the flagged *share* will not match — that difference is a real
property of the classifiers, which is exactly why it's shown here instead of
being silently absorbed into the row count.

In [0]:
region_counts = df["region"].value_counts().reindex(regions)

fig = go.Figure(go.Bar(
    x=region_counts.index,
    y=region_counts.values,
    marker=dict(color=[REGION_COLOR[r] for r in region_counts.index],
                cornerradius=4),
    text=region_counts.values,
    textposition="outside",
    hovertemplate="<b>%{x}</b><br>%{y} songs<extra></extra>",
    showlegend=False,
))
fig.update_layout(
    title=f"Classified songs per region — {LABEL}<br>"
          f"<sup>{len(df)} songs total; unclassified already removed in 05.x</sup>",
    yaxis_title="songs", xaxis_title=None, bargap=0.35,
    height=420, width=900, **LAYOUT,
)
fig.update_yaxes(gridcolor="#eceae5", zeroline=False)
fig.update_xaxes(showgrid=False)
fig.show()

# Confidence is carried as data, so report it rather than let it hide in the counts.
conf = (
    df.groupby("region")
      .agg(songs=("spotify_uri", "size"),
           median_dominant_score=("dominant_score", "median"),
           pct_low_confidence=("low_confidence", lambda s: s.mean() * 100))
      .reindex(regions)
)
print(f"Overall low_confidence share: {df['low_confidence'].mean():.1%} "
      f"(bar = 0.30, shared with the other fork)")
display(conf.round(2))

### 2. Dominant emotion mix

Which label wins outright per song, counted overall and split by region. Counts
are fork-local — the label sets don't line up, so a bar here has no counterpart
in the sibling notebook.

In [0]:
dom = df[DOMINANT_COL].value_counts()
dom = dom[~dom.index.isin(EXCLUDE)]

fig = go.Figure(go.Bar(
    x=dom.values, y=dom.index, orientation="h",
    marker=dict(color="#2a78d6", cornerradius=4),
    text=dom.values, textposition="outside",
    hovertemplate="<b>%{y}</b><br>%{x} songs<extra></extra>",
))
fig.update_layout(
    title=f"Dominant emotion across all songs — {LABEL}<br>"
          f"<sup>column charted: <code>{DOMINANT_COL}</code></sup>",
    xaxis_title="songs", yaxis=dict(autorange="reversed"),
    height=max(360, 26 * len(dom) + 140), width=860, bargap=0.35, **LAYOUT,
)
fig.update_xaxes(gridcolor="#eceae5", zeroline=False)
fig.update_yaxes(showgrid=False)
fig.show()

# Share of each region's songs, so a large region doesn't dominate by size alone
mix = (
    pd.crosstab(df["region"], df[DOMINANT_COL], normalize="index")
      .reindex(regions)
      .drop(columns=[c for c in EXCLUDE if c in df[DOMINANT_COL].unique()], errors="ignore")
)
mix = mix[dom.index[: min(12, len(dom))]]
display((mix * 100).round(1))

### 3. Regional emotion profile — raw mean scores *(fork-local)*

Mean score per region per emotion, on this fork's native scale. Useful for
reading *within* the grid (which emotion runs hottest in a region), **not** for
comparing a cell against the sibling notebook's same-named cell.

In [0]:
heat = df.groupby("region")[charted_cols].mean().reindex(regions)
heat.columns = charted_names

fig = px.imshow(
    heat.T,
    labels=dict(x="", y="", color="mean score"),
    color_continuous_scale=SEQ_BLUE,
    aspect="auto",
    text_auto=".2f",
)
fig.update_traces(
    textfont_size=11,
    hovertemplate="<b>%{x}</b> · %{y}<br>mean score %{z:.3f}<extra></extra>",
    xgap=2, ygap=2,   # surface gap between cells
)
fig.update_layout(
    title=f"Mean emotion score by region — {LABEL}<br>"
          f"<sup>Fork-local scale. Compare cells within this grid, not against 06.x's other fork.</sup>",
    height=60 + 34 * len(charted_names) + 140, width=1000,
    coloraxis_colorbar=dict(title="mean", thickness=12, len=0.7),
    **LAYOUT,
)
fig.show()
display(heat.round(3))

### 4. Regional character — z-scored within this fork *(cross-fork readable)*

Each emotion column is standardised across regions inside this fork
(`(region mean − global mean) / std across regions`). That strips out both the
overall scale and each label's own baseline popularity, leaving a pure
"how unusual is this region on this emotion" reading.

This is the chart to hold next to the sibling notebook's version of it. A value
of +1.5 means the same thing in both — *1.5 standard deviations above this
classifier's own regional baseline* — even though the underlying raw scores are
on incomparable scales. Diverging blue↔red with a gray midpoint, because zero is
a real and meaningful centre here.

In [0]:
region_means = df.groupby("region")[charted_cols].mean().reindex(regions)
z = (region_means - region_means.mean()) / region_means.std(ddof=0)
z.columns = charted_names
z = z.fillna(0)

lim = float(np.abs(z.values).max())

fig = px.imshow(
    z.T,
    labels=dict(x="", y="", color="z-score"),
    color_continuous_scale=DIVERGING,
    zmin=-lim, zmax=lim,
    aspect="auto",
    text_auto=".1f",
)
fig.update_traces(
    textfont_size=11,
    hovertemplate="<b>%{x}</b> · %{y}<br>%{z:+.2f} SD vs this fork's regional mean<extra></extra>",
    xgap=2, ygap=2,
)
fig.update_layout(
    title=f"Regional emotion character — {LABEL}<br>"
          f"<sup>Standard deviations from this fork's own regional mean. "
          f"Red = distinctively high, blue = distinctively low. Comparable across forks.</sup>",
    height=60 + 34 * len(charted_names) + 150, width=1000,
    coloraxis_colorbar=dict(title="SD", thickness=12, len=0.7),
    **LAYOUT,
)
fig.show()
display(z.round(2))

### 5. Where each emotion peaks

Top 5 regions per emotion. Read the *ordering* of the bars; the heights are on
the fork-local scale again.

In [0]:
n = len(charted_cols)
ncols = 3
nrows = -(-n // ncols)

fig = make_subplots(
    rows=nrows, cols=ncols,
    subplot_titles=[c.replace("emotion_", "") for c in charted_cols],
    vertical_spacing=0.09 if nrows <= 4 else 0.05,
    horizontal_spacing=0.07,
)

for idx, col in enumerate(charted_cols):
    r, c = idx // ncols + 1, idx % ncols + 1
    top = df.groupby("region")[col].mean().sort_values(ascending=False).head(5)
    fig.add_trace(
        go.Bar(
            x=top.index, y=top.values,
            marker=dict(color=[REGION_COLOR[i] for i in top.index], cornerradius=4),
            hovertemplate="<b>%{x}</b><br>mean %{y:.3f}<extra></extra>",
            showlegend=False,
        ),
        row=r, col=c,
    )

fig.update_annotations(font_size=13)
fig.update_yaxes(gridcolor="#eceae5", zeroline=False, title=None)
fig.update_xaxes(showgrid=False, tickangle=-35, tickfont_size=10)
fig.update_layout(
    title_text=f"Top 5 regions per emotion — {LABEL}<br>"
               f"<sup>Fork-local mean scores; region colours are fixed across every chart here.</sup>",
    height=260 * nrows + 120, width=1150, bargap=0.35, **LAYOUT,
)
fig.show()

### 6. Score spread within each region

Means hide bimodality — a region can average mid on an emotion because every
song is mid, or because half its songs are extreme. Box plots separate those.

In [0]:
melted = (
    df[["region"] + charted_cols]
      .melt(id_vars="region", var_name="emotion", value_name="score")
)
melted["emotion"] = melted["emotion"].str.replace("emotion_", "", regex=False)

fig = px.box(
    melted, x="emotion", y="score", color="region",
    color_discrete_map=REGION_COLOR,
    category_orders={"emotion": charted_names, "region": regions},
    points=False,
)
fig.update_traces(line_width=1.5, marker_size=4)
fig.update_layout(
    title=f"Score distribution by emotion and region — {LABEL}<br>"
          f"<sup>Box = IQR, line = median. Fork-local scale.</sup>",
    xaxis_title=None, yaxis_title="score",
    boxmode="group", height=560, width=max(1000, 78 * len(charted_names)),
    legend=dict(orientation="h", y=-0.28, title=None),
    **LAYOUT,
)
fig.update_yaxes(gridcolor="#eceae5", zeroline=False)
fig.update_xaxes(showgrid=False, tickangle=-30)
fig.show()

### 7. Table view

Every chart above has a numeric counterpart here — required so nothing in this
notebook is readable by colour alone.

In [0]:
summary = pd.concat(
    {
        "mean score": df.groupby("region")[charted_cols].mean().reindex(regions).T,
    },
    axis=1,
)
summary.index = charted_names
display(summary.round(3))

print("\nGlobal ranking of emotions in this fork:")
display(
    df[emotion_cols].mean().sort_values(ascending=False)
      .rename("mean score").to_frame().round(3)
)

### 8. The neutral label — GoEmotions only

No zero-shot equivalent exists, so this section has no counterpart in `06.1`.
A high neutral share means the classifier read a lot of these lyrics as
affectively flat, which is itself a finding about the taxonomy fit: GoEmotions
was trained on Reddit comments, not song lyrics.

In [0]:
neutral_by_region = (
    df.groupby("region")
      .agg(mean_neutral=("emotion_neutral", "mean"),
           pct_dominant_neutral=("dominant_emotion",
                                 lambda s: (s == "neutral").mean() * 100))
      .reindex(regions)
)

fig = go.Figure(go.Bar(
    x=neutral_by_region.index,
    y=neutral_by_region["pct_dominant_neutral"],
    marker=dict(color=[REGION_COLOR[r] for r in neutral_by_region.index],
                cornerradius=4),
    text=neutral_by_region["pct_dominant_neutral"].round(1),
    texttemplate="%{text}%", textposition="outside",
    hovertemplate="<b>%{x}</b><br>%{y:.1f}% of songs read as neutral<extra></extra>",
))
fig.update_layout(
    title="Share of songs GoEmotions calls <b>neutral</b><br>"
          "<sup>No zero-shot counterpart — 04.1's taxonomy has no neutral label</sup>",
    yaxis_title="% of region's songs", xaxis_title=None,
    height=430, width=900, bargap=0.35, **LAYOUT,
)
fig.update_yaxes(gridcolor="#eceae5", zeroline=False)
fig.update_xaxes(showgrid=False)
fig.show()

display(neutral_by_region.round(2))